# Partie 4 — Entraînement & optimisation des modèles

**Projet :** prédiction du churn client (Telco Customer Churn) — groupe de 5
**Responsable :** Membre 4
**Livrable attendu :** modèles entraînés + tableau comparatif des expériences

## Objectif de cette partie

Les parties 1 à 3 ont fourni un dataset compris, exploré et préparé. Ici, on **entraîne plusieurs modèles de classification**, on les compare de façon **honnête** (sans fuite de données) et on **optimise** les meilleurs. L'évaluation finale détaillée (matrices de confusion, interprétation) est le travail de la partie 5.

## Ce qu'on retient des parties précédentes

| Constat | Conséquence pour la modélisation |
|---|---|
| 26,5 % de churn seulement (classes déséquilibrées) | l'accuracy est trompeuse : un modèle « toujours No » obtient 73,5 %. On suit surtout **Recall, F1 et ROC-AUC** de la classe churn |
| Contrat, ancienneté, fibre, chèque électronique = facteurs de risque | on pourra vérifier que les modèles retrouvent ces variables |
| `tenure`, `TotalCharges`, `MonthlyCharges` très corrélées | attention aux modèles linéaires (régularisation) |
| Découpage train/test déjà fait (80/20, stratifié, `random_state=42`) | on **réutilise exactement le même découpage** ; le test n'est touché qu'à la toute fin |

## Plan du notebook

1. **Mise en place** : configuration, chargement des données, reconstruction du pipeline de prétraitement
2. Stratégie de validation et modèle de référence
3. Entraînement de plusieurs modèles (Logistic Regression, Decision Tree, Random Forest…)
4. Gestion du déséquilibre des classes
5. Optimisation des hyperparamètres
6. Tableau comparatif des expériences et sauvegarde des modèles

## 1. Mise en place

### 1.1 Imports et configuration

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
import imblearn
import joblib

# Graine unique pour toute la partie 4 : mêmes résultats à chaque exécution
SEED = 42

# Dossier lu par la partie 4 (produit par la partie 3) et dossiers de sortie
OUT_DIR = Path("outputs")
MODELS_DIR = Path("../models")
REPORTS_DIR = Path("../reports")

pd.set_option("display.max_columns", 60)
print(f"Python {sys.version.split()[0]} | pandas {pd.__version__} | numpy {np.__version__}")
print(f"scikit-learn {sklearn.__version__} | imbalanced-learn {imblearn.__version__}")

Python 3.13.7 | pandas 3.1.1 | numpy 2.5.4
scikit-learn 1.9.1 | imbalanced-learn 0.14.2


**Pourquoi afficher les versions ?** Un modèle sauvegardé avec `joblib` ne se recharge correctement qu'avec la même version de scikit-learn. Les versions sont fixées dans `requirements.txt` ; cette cellule permet de vérifier qu'on travaille avec le même environnement que le reste du groupe.

### 1.2 Chargement des données

On part de `telco_clean_features.csv` : les données **nettoyées avec les nouvelles variables, mais avant encodage et mise à l'échelle** (produit par la partie 3).

**Pourquoi ne pas utiliser directement `train_processed.csv` ?** Dans ces fichiers, l'imputation et la standardisation ont déjà été apprises sur *tout* le train. Si on fait ensuite une validation croisée dessus, les statistiques (moyenne, écart-type) de chaque pli de validation ont déjà « fuité » dans l'entraînement : c'est une fuite légère, mais réelle. En repartant des données avant encodage, on pourra remettre le prétraitement **à l'intérieur** de chaque pli de validation croisée.

In [2]:
df = pd.read_csv(OUT_DIR / "telco_clean_features.csv")
print("Dimensions :", df.shape)
print("Valeurs manquantes par colonne (uniquement celles > 0) :")
print(df.isna().sum()[lambda s: s > 0])
df.head(3)

Dimensions : (7043, 45)
Valeurs manquantes par colonne (uniquement celles > 0) :
Series([], dtype: int64)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,tenure_group,is_new_customer,tenure_years,has_internet,has_phone,n_security_services,n_streaming_services,n_services_total,has_protection_bundle,internet_no_protection,avg_monthly_spend,charge_delta,charge_ratio,charge_per_service,log_total_charges,is_month_to_month,contract_length_m,auto_payment,is_electronic_check,has_family,is_alone,fiber_month_to_month,new_and_mtm,senior_alone
0,7590-VHVEG,Female,No,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,0,0-6m,1,0.083333,1,0,1,0,2,0,0,29.850000,0.000000,1.000000,14.9250,3.429137,1,1,0,1,1,0,0,1,0
1,5575-GNVDE,Male,No,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,0,25-48m,0,2.833333,1,1,2,0,4,0,0,55.573529,1.376471,1.024768,14.2375,7.544597,0,12,0,0,0,1,0,0,0
2,3668-QPYBK,Male,No,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,1,0-6m,1,0.166667,1,1,2,0,4,0,0,54.075000,-0.225000,0.995839,13.4625,4.692723,1,1,0,0,0,1,0,1,0


### 1.3 Découpage train / test (identique à la partie 3)

Mêmes paramètres que la partie 3 : 80 % / 20 %, stratifié sur `Churn`, `random_state=42`. Le **jeu de test est mis de côté** : il ne servira qu'à la fin, pour évaluer le modèle retenu. Toutes les décisions (choix du modèle, réglage des hyperparamètres) se feront uniquement avec le train.

In [3]:
from sklearn.model_selection import train_test_split

TARGET = "Churn"
X = df.drop(columns=[TARGET, "customerID"])   # customerID est un identifiant, pas une variable explicative
y = df[TARGET]
ids = df["customerID"]

X_train, X_test, y_train, y_test, id_train, id_test = train_test_split(
    X, y, ids, test_size=0.20, stratify=y, random_state=SEED)

print(f"Train : {X_train.shape} | churn = {y_train.mean():.2%}")
print(f"Test  : {X_test.shape}  | churn = {y_test.mean():.2%}")

Train : (5634, 43) | churn = 26.54%
Test  : (1409, 43)  | churn = 26.54%


### 1.4 Vérification : est-ce le même découpage que la partie 3 ?

Si le groupe n'utilise pas les mêmes clients dans le train et le test, les résultats des parties 4 et 5 ne seront pas comparables. On compare donc les identifiants avec les fichiers produits par la partie 3.

In [4]:
ref_train = pd.read_csv(OUT_DIR / "train_processed.csv", usecols=["customerID", "Churn"])
ref_test = pd.read_csv(OUT_DIR / "test_processed.csv", usecols=["customerID", "Churn"])

assert set(id_train) == set(ref_train["customerID"]), "Le train n'est pas celui de la partie 3"
assert set(id_test) == set(ref_test["customerID"]), "Le test n'est pas celui de la partie 3"
assert set(id_train).isdisjoint(set(id_test)), "Des clients sont à la fois dans le train et le test"
print("OK : train et test identiques à ceux de la partie 3 "
      f"({len(id_train)} + {len(id_test)} clients, aucun client en commun).")

OK : train et test identiques à ceux de la partie 3 (5634 + 1409 clients, aucun client en commun).


**Résultat :** on travaille bien sur les mêmes clients que le reste du groupe, avec la même proportion de churn (26,54 %) dans le train et dans le test.

### 1.5 Reconstruction du pipeline de prétraitement

On reprend **exactement les mêmes choix que la partie 3** (mêmes colonnes, mêmes imputations, même encodage), mais sous forme d'une fonction `make_preprocessor()` qui renvoie un pipeline **neuf et non ajusté**.

**Pourquoi une fonction ?** En validation croisée, il faut ré-apprendre le prétraitement sur chaque pli d'entraînement. Chaque pli reçoit donc son propre pipeline vierge, qui n'a jamais vu les données de validation.

Rappel des choix de la partie 3 :
- **Numériques** : imputation par la médiane, puis standardisation (utile aux modèles sensibles aux échelles, comme la régression logistique)
- **Binaires 0/1** : conservées telles quelles
- **`tenure_group`** : encodage ordinal (les tranches d'ancienneté sont ordonnées)
- **Nominales** : one-hot (`drop="if_binary"` garde une seule colonne pour les variables Yes/No)
- Les 6 variables exactement redondantes (`has_phone`, `has_internet`, `is_electronic_check`, `is_month_to_month`, `tenure_years`, `is_alone`) ne sont pas utilisées : elles sont ignorées par `remainder="drop"`.

In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

numeric_features = ["tenure", "MonthlyCharges", "TotalCharges", "log_total_charges",
                    "avg_monthly_spend", "charge_delta", "charge_ratio", "charge_per_service",
                    "n_security_services", "n_streaming_services", "n_services_total", "contract_length_m"]

binary_passthrough = ["is_new_customer", "has_protection_bundle", "internet_no_protection",
                      "auto_payment", "has_family", "fiber_month_to_month", "new_and_mtm", "senior_alone"]

ordinal_features = ["tenure_group"]
ordinal_categories = [["0-6m", "7-12m", "13-24m", "25-48m", "49-72m"]]

nominal_features = ["gender", "SeniorCitizen", "Partner", "Dependents", "PhoneService", "MultipleLines",
                    "InternetService", "OnlineSecurity", "OnlineBackup", "DeviceProtection", "TechSupport",
                    "StreamingTV", "StreamingMovies", "Contract", "PaperlessBilling", "PaymentMethod"]

redundant_features = ["has_phone", "has_internet", "is_electronic_check",
                      "is_month_to_month", "tenure_years", "is_alone"]


def make_preprocessor():
    """Renvoie un prétraitement NEUF (non ajusté), identique à celui de la partie 3."""
    numeric_pipe = Pipeline([("imputer", SimpleImputer(strategy="median")),
                             ("scaler", StandardScaler())])
    ordinal_pipe = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                             ("encoder", OrdinalEncoder(categories=ordinal_categories))])
    nominal_pipe = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                             ("encoder", OneHotEncoder(drop="if_binary", handle_unknown="infrequent_if_exist",
                                                       min_frequency=0.01, sparse_output=False))])
    return ColumnTransformer(
        transformers=[("num", numeric_pipe, numeric_features),
                      ("bin", "passthrough", binary_passthrough),
                      ("ord", ordinal_pipe, ordinal_features),
                      ("nom", nominal_pipe, nominal_features)],
        remainder="drop",
        verbose_feature_names_out=False,
    ).set_output(transform="pandas")


# Garde-fou : toutes les colonnes de X sont soit utilisées, soit explicitement écartées
used = numeric_features + binary_passthrough + ordinal_features + nominal_features
assert len(used) == len(set(used)), "Colonne utilisée deux fois"
assert set(used) | set(redundant_features) == set(X.columns), set(X.columns) ^ (set(used) | set(redundant_features))
print(f"{len(used)} variables utilisées, {len(redundant_features)} écartées (redondantes).")

37 variables utilisées, 6 écartées (redondantes).


### 1.6 Vérification : le pipeline reconstruit redonne-t-il les fichiers de la partie 3 ?

On ajuste le nouveau pipeline sur le train et on compare le résultat, valeur par valeur, avec `train_processed.csv` et `test_processed.csv` (produits par la partie 3). Si tout est identique, on est sûr de ne pas avoir changé le prétraitement en le reconstruisant.

In [6]:
prep_check = make_preprocessor()
Xtr_p = prep_check.fit_transform(X_train)   # fit sur le train uniquement
Xte_p = prep_check.transform(X_test)

feature_names_p3 = pd.read_csv(OUT_DIR / "feature_names.csv")["feature"].tolist()
assert list(Xtr_p.columns) == feature_names_p3, "Les noms/ordre des colonnes diffèrent de la partie 3"

for nom, mine, fichier, ids_ in [("train", Xtr_p, "train_processed.csv", id_train),
                                 ("test", Xte_p, "test_processed.csv", id_test)]:
    ref = pd.read_csv(OUT_DIR / fichier).set_index("customerID").loc[ids_.values]   # même ordre de clients
    ecart = np.abs(mine.to_numpy() - ref[feature_names_p3].to_numpy()).max()
    assert ecart < 1e-6, f"{nom} : écart max {ecart}"
    print(f"{nom:5s}: {mine.shape} - écart maximal avec la partie 3 = {ecart:.2e}")

print("OK : le pipeline reconstruit est identique à celui de la partie 3.")

train: (5634, 58) - écart maximal avec la partie 3 = 0.00e+00
test : (1409, 58) - écart maximal avec la partie 3 = 0.00e+00
OK : le pipeline reconstruit est identique à celui de la partie 3.


**Résultat :** le pipeline est identique à celui de la partie 3 (mêmes 58 variables, mêmes valeurs). La différence est dans l'usage : à partir de maintenant, il sera appelé *à l'intérieur* de chaque pli de validation croisée, ce qui garantit qu'aucune information du pli de validation n'est utilisée pour apprendre l'imputation ou la standardisation.

**Fin de l'étape 1.** Prochaine étape : définir la stratégie de validation et le modèle de référence.

## 2. Stratégie de validation et modèle de référence

### 2.1 Pourquoi une validation croisée ?

Pour **comparer des modèles et régler leurs hyperparamètres**, il faut pouvoir les juger sur des données qu'ils n'ont pas vues à l'entraînement. Mais on ne peut pas utiliser le jeu de test pour ça : si on choisit le modèle qui obtient le meilleur score sur le test, ce score devient optimiste (le test a influencé notre choix) et ne mesure plus la performance réelle.

La solution est la **validation croisée** *à l'intérieur du train* :
1. on coupe le train en 5 parties (« plis ») ;
2. on entraîne sur 4 plis et on évalue sur le 5ᵉ, puis on recommence 5 fois en changeant le pli de validation ;
3. on fait la **moyenne** des 5 scores (et on regarde leur écart-type pour mesurer la stabilité).

Le jeu de test, lui, reste intact jusqu'à la toute fin.

### 2.2 Découpage en plis

On utilise `StratifiedKFold` :
- **stratifié** : chaque pli garde environ 26,5 % de churn, ce qui est essentiel avec des classes déséquilibrées ;
- `shuffle=True` avec `random_state=SEED` : les clients sont mélangés avant la découpe, de façon reproductible.

In [7]:
from sklearn.model_selection import StratifiedKFold

N_SPLITS = 5
cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

# Contrôle : taille et taux de churn de chaque pli de validation
lignes = []
for i, (idx_tr, idx_val) in enumerate(cv.split(X_train, y_train), start=1):
    lignes.append({"pli": i, "n_entrainement": len(idx_tr), "n_validation": len(idx_val),
                   "churn_validation": f"{y_train.iloc[idx_val].mean():.2%}"})
pd.DataFrame(lignes).set_index("pli")

,n_entrainement,n_validation,churn_validation
pli,,,
1,4507,1127,26.53%
2,4507,1127,26.53%
3,4507,1127,26.53%
4,4507,1127,26.53%
5,4508,1126,26.55%


### 2.3 Métriques suivies

Avec 26,5 % de churn, l'**accuracy** seule est trompeuse : un modèle qui répond toujours « le client reste » aurait 73,5 % d'accuracy tout en ne détectant **aucun** client qui part. On suit donc plusieurs métriques, en considérant « churn » comme la classe positive :

| Métrique | Question à laquelle elle répond |
|---|---|
| **Recall** (rappel) | parmi les clients qui partent vraiment, combien en détecte-t-on ? |
| **Precision** | parmi les clients signalés comme à risque, combien partent vraiment ? |
| **F1-score** | compromis entre precision et recall |
| **ROC-AUC** | capacité à classer les clients du plus au moins à risque, indépendamment du seuil de décision |
| Accuracy | à titre indicatif seulement |

**Métrique de décision : le F1-score de la classe churn**, avec le Recall en seconde lecture. Le document de la partie 1 indique que rater un client qui part coûte plus cher que lancer une action de fidélisation inutile : le recall compte donc beaucoup, mais le F1 évite de gagner du recall en signalant tout le monde.

In [8]:
from sklearn.metrics import make_scorer, precision_score, recall_score

# zero_division=0 : si un modèle ne prédit jamais « churn », precision = 0 (au lieu d'un avertissement)
scoring = {
    "accuracy": "accuracy",
    "precision": make_scorer(precision_score, zero_division=0),
    "recall": make_scorer(recall_score, zero_division=0),
    "f1": "f1",
    "roc_auc": "roc_auc",
}
print("Métriques calculées à chaque pli :", list(scoring))

Métriques calculées à chaque pli : ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']


### 2.4 Fonction d'expérience et journal des résultats

Le TP demande de **consigner les paramètres et les résultats de chaque expérience**. On crée :
- une liste `experiences` qui garde une ligne par expérience ;
- une fonction `evaluer(nom, modele, ...)` qui place le modèle derrière le prétraitement dans un `Pipeline`, lance la validation croisée sur le train, et enregistre les moyennes et écarts-types.

Le prétraitement est **dans** le pipeline : il est donc ré-appris sur les 4 plis d'entraînement à chaque tour, sans jamais voir le pli de validation.

In [9]:
import time
from imblearn.pipeline import Pipeline as ImbPipeline   # comme Pipeline, mais accepte un rééchantillonneur (SMOTE)
from sklearn.model_selection import cross_validate

experiences = []   # journal : une ligne par expérience


def evaluer(nom, modele, remarque="", sampler=None):
    """Validation croisée (prétraitement, et éventuel rééchantillonnage, inclus dans chaque pli) + journal.

    Le rééchantillonneur (ex. SMOTE) n'agit qu'à l'entraînement de chaque pli : le pli de validation
    garde sa vraie répartition de classes."""
    etapes = [("prep", make_preprocessor())]
    if sampler is not None:
        etapes.append(("reechantillonnage", sampler))
    etapes.append(("modele", modele))
    pipe = ImbPipeline(etapes)
    debut = time.time()
    res = cross_validate(pipe, X_train, y_train, cv=cv, scoring=scoring, n_jobs=1)
    ligne = {"experience": nom}
    for m in scoring:
        ligne[f"{m}_moy"] = res[f"test_{m}"].mean()
        ligne[f"{m}_std"] = res[f"test_{m}"].std()
    ligne["temps_s"] = time.time() - debut
    ligne["parametres"] = str(modele.get_params(deep=False)) + (f" | {sampler}" if sampler is not None else "")
    ligne["remarque"] = remarque
    experiences.append(ligne)
    return ligne


def tableau():
    """Tableau lisible du journal, trié par F1 décroissant."""
    cols = ["accuracy_moy", "precision_moy", "recall_moy", "f1_moy", "f1_std", "roc_auc_moy", "temps_s"]
    return pd.DataFrame(experiences).set_index("experience")[cols].sort_values("f1_moy", ascending=False).round(3)

### 2.5 Modèle de référence (baseline)

Un modèle de référence donne le **score minimal à battre** : si un vrai modèle ne fait pas mieux, il n'apporte rien. On prend le classifieur le plus simple possible, `DummyClassifier(strategy="most_frequent")`, qui répond toujours « le client reste » (la classe majoritaire).

In [10]:
from sklearn.dummy import DummyClassifier

evaluer("Baseline - toujours 'reste'", DummyClassifier(strategy="most_frequent"),
        remarque="classe majoritaire")
tableau()

,accuracy_moy,precision_moy,recall_moy,f1_moy,f1_std,roc_auc_moy,temps_s
experience,,,,,,,
Baseline - toujours 'reste',0.735,0.0,0.0,0.0,0.0,0.5,4.892


**Lecture du résultat :**
- **Accuracy = 73,5 %** : ça ressemble à un bon score, mais c'est uniquement la proportion de clients qui restent. Le modèle ne fait que répéter la classe majoritaire.
- **Recall = 0 et F1 = 0** : il ne détecte aucun des clients qui partent, ce qui est justement le but du projet.
- **ROC-AUC = 0,5** : c'est le niveau du hasard, le modèle ne sait pas classer les clients.

Cela confirme le choix des métriques : l'accuracy seule aurait fait passer ce modèle inutile pour un bon modèle. **Tout vrai modèle devra faire mieux que F1 = 0 et ROC-AUC = 0,5**, et surtout se juger sur le F1 et le recall de la classe churn.

**Fin de l'étape 2.** Prochaine étape : entraîner plusieurs modèles (Logistic Regression, Decision Tree, Random Forest) avec leurs paramètres par défaut.

## 3. Entraînement de plusieurs modèles (paramètres par défaut)

On entraîne d'abord trois modèles classiques **avec leurs réglages par défaut**, pour voir ce que chacun donne « tel quel » avant toute optimisation. Ils sont évalués par la même validation croisée que le modèle de référence, donc directement comparables.

| Modèle | Principe | Points forts | Points faibles |
|---|---|---|---|
| **Régression logistique** | combine les variables avec des poids pour estimer une probabilité de churn | simple, rapide, interprétable (un poids par variable) | ne capte que des effets linéaires ; sensible aux variables corrélées |
| **Arbre de décision** | pose une suite de questions « oui/non » (ex. contrat mensuel ? ancienneté < 6 mois ?) | très lisible, capte les interactions | tend à **sur-apprendre** (il mémorise le train) quand on ne limite pas sa profondeur |
| **Forêt aléatoire** | moyenne de nombreux arbres entraînés sur des échantillons différents | robuste, performante en général, limite le sur-apprentissage | moins lisible, plus lente |

À chaque fois, `random_state=SEED` garantit des résultats reproductibles.

### 3.1 Régression logistique

On fixe `max_iter=1000` (au lieu de 100 par défaut) uniquement pour laisser l'algorithme converger ; les autres paramètres sont ceux par défaut, notamment la régularisation `C=1.0`.

In [11]:
from sklearn.linear_model import LogisticRegression

evaluer("Régression logistique (défaut)", LogisticRegression(max_iter=1000, random_state=SEED))
tableau()

,accuracy_moy,precision_moy,recall_moy,f1_moy,f1_std,roc_auc_moy,temps_s
experience,,,,,,,
Régression logistique (défaut),0.808,0.673,0.536,0.596,0.033,0.849,6.169
Baseline - toujours 'reste',0.735,0.000,0.000,0.000,0.000,0.500,4.892


### 3.2 Arbre de décision

Sans aucune limite de profondeur : l'arbre peut grandir jusqu'à classer parfaitement chaque client du train.

In [12]:
from sklearn.tree import DecisionTreeClassifier

evaluer("Arbre de décision (défaut)", DecisionTreeClassifier(random_state=SEED))
tableau()

,accuracy_moy,precision_moy,recall_moy,f1_moy,f1_std,roc_auc_moy,temps_s
experience,,,,,,,
Régression logistique (défaut),0.808,0.673,0.536,0.596,0.033,0.849,6.169
Arbre de décision (défaut),0.728,0.489,0.512,0.500,0.014,0.660,6.435
Baseline - toujours 'reste',0.735,0.000,0.000,0.000,0.000,0.500,4.892


### 3.3 Forêt aléatoire

Réglages par défaut : 100 arbres, sans limite de profondeur.

In [13]:
from sklearn.ensemble import RandomForestClassifier

evaluer("Forêt aléatoire (défaut)", RandomForestClassifier(random_state=SEED))
tableau()

,accuracy_moy,precision_moy,recall_moy,f1_moy,f1_std,roc_auc_moy,temps_s
experience,,,,,,,
Régression logistique (défaut),0.808,0.673,0.536,0.596,0.033,0.849,6.169
Forêt aléatoire (défaut),0.791,0.638,0.495,0.557,0.026,0.828,23.407
Arbre de décision (défaut),0.728,0.489,0.512,0.500,0.014,0.660,6.435
Baseline - toujours 'reste',0.735,0.000,0.000,0.000,0.000,0.500,4.892


### 3.4 Vérification du sur-apprentissage : score sur le train vs score en validation

Un modèle qui **sur-apprend** a un excellent score sur les données qu'il a vues et un score bien plus faible sur des données nouvelles. On compare donc, pour chaque modèle, le F1 mesuré sur les plis d'entraînement et celui mesuré sur les plis de validation. Un grand écart signale du sur-apprentissage.

In [14]:
modeles_defaut = {
    "Régression logistique": LogisticRegression(max_iter=1000, random_state=SEED),
    "Arbre de décision": DecisionTreeClassifier(random_state=SEED),
    "Forêt aléatoire": RandomForestClassifier(random_state=SEED),
}
lignes = []
for nom, m in modeles_defaut.items():
    pipe = Pipeline([("prep", make_preprocessor()), ("modele", m)])
    r = cross_validate(pipe, X_train, y_train, cv=cv, scoring="f1", return_train_score=True, n_jobs=1)
    lignes.append({"modèle": nom, "F1 train": r["train_score"].mean(), "F1 validation": r["test_score"].mean()})
ecarts = pd.DataFrame(lignes).set_index("modèle")
ecarts["écart"] = ecarts["F1 train"] - ecarts["F1 validation"]
ecarts.round(3)

,F1 train,F1 validation,écart
modèle,,,
Régression logistique,0.603,0.596,0.007
Arbre de décision,0.997,0.500,0.497
Forêt aléatoire,0.997,0.557,0.439


### 3.5 Lecture des résultats

| Modèle (défaut) | F1 | Recall | ROC-AUC | F1 train → validation |
|---|---|---|---|---|
| Régression logistique | **0,596** | **0,536** | **0,849** | 0,603 → 0,596 |
| Forêt aléatoire | 0,557 | 0,495 | 0,828 | 0,997 → 0,557 |
| Arbre de décision | 0,500 | 0,512 | 0,660 | 0,997 → 0,500 |
| Référence (toujours « reste ») | 0 | 0 | 0,500 | — |

**Ce qu'on observe**
1. **Tous les modèles battent la référence**, ce qui montre que les variables contiennent bien de l'information sur le churn.
2. **La régression logistique est la meilleure** sur les trois métriques de décision, et la plus **stable** : son score est quasi identique sur le train et en validation (écart de 0,007). Elle ne sur-apprend pas.
3. **L'arbre et la forêt sur-apprennent fortement** : F1 de 0,997 sur le train (ils retiennent presque par cœur les clients vus) mais 0,500 et 0,557 en validation. Sans limite de profondeur, ils ont mémorisé le bruit du train au lieu d'apprendre des règles générales.
4. L'arbre seul est même **moins bon en accuracy (72,8 %) que le modèle de référence (73,5 %)** : un modèle qui sur-apprend peut faire pire qu'une règle triviale.
5. **Le recall reste modeste (≈ 50 %)** : avec les réglages par défaut, on rate environ un client sur deux qui part. C'est la principale piste d'amélioration.

**Ce qu'on en déduit pour la suite**
- Pour l'arbre et la forêt, le problème n'est pas le modèle mais ses **réglages** : il faut limiter la complexité (profondeur maximale, taille minimale des feuilles). C'est typiquement le rôle de l'optimisation d'hyperparamètres (étape 5).
- Pour améliorer le recall sur tous les modèles, on va tester la **gestion du déséquilibre des classes** (étape 4) : donner plus de poids à la classe « churn » ou rééquilibrer les données.

**Fin de l'étape 3.** Prochaine étape : gérer le déséquilibre des classes.

## 4. Gestion du déséquilibre des classes

**Le problème.** Seulement 26,5 % des clients partent. Un modèle qui cherche à minimiser le nombre total d'erreurs est naturellement tenté de privilégier la classe majoritaire (« reste »), car c'est là qu'il se trompe le moins. Résultat observé à l'étape 3 : un recall d'environ 50 %, donc un client sur deux qui part n'est pas détecté.

**Deux approches testées**

| Approche | Principe | Où ça agit |
|---|---|---|
| **Pondération des classes** (`class_weight="balanced"`) | une erreur sur un client qui part « coûte » plus cher qu'une erreur sur un client qui reste (ici ≈ 2,8 fois plus : 73,5 / 26,5) | dans la fonction de coût du modèle ; les données ne changent pas |
| **SMOTE** | fabrique de nouveaux clients « churn » synthétiques, en interpolant entre des clients churn voisins, jusqu'à équilibrer les deux classes | sur les données d'entraînement uniquement |

**Précautions**
- Le rééchantillonnage (SMOTE) est placé **dans le pipeline** : à chaque pli de validation croisée, il n'est appliqué qu'aux 4 plis d'entraînement. Le pli de validation garde sa répartition réelle (26,5 % de churn), sinon le score serait faussé.
- SMOTE agit ici après l'encodage : il interpole aussi les colonnes 0/1 issues du one-hot, qui peuvent prendre des valeurs intermédiaires (ex. 0,4). C'est une approximation courante ; la variante `SMOTENC` gère mieux les variables catégorielles mais demande de restructurer le pipeline. On commence par SMOTE standard.

**Ce qu'on s'attend à voir** : le **recall monte** (on détecte plus de clients qui partent) et la **precision baisse** (on signale aussi plus de clients qui ne partiront pas), donc l'**accuracy baisse** : ce n'est pas un défaut, c'est le prix de la détection. Le F1 et le ROC-AUC diront si le compromis est globalement meilleur.

### 4.1 Régression logistique pondérée

In [15]:
evaluer("Régression logistique + class_weight",
        LogisticRegression(max_iter=1000, class_weight="balanced", random_state=SEED),
        remarque="pondération des classes")
tableau()

,accuracy_moy,precision_moy,recall_moy,f1_moy,f1_std,roc_auc_moy,temps_s
experience,,,,,,,
Régression logistique + class_weight,0.752,0.521,0.792,0.628,0.023,0.848,6.248
Régression logistique (défaut),0.808,0.673,0.536,0.596,0.033,0.849,6.169
Forêt aléatoire (défaut),0.791,0.638,0.495,0.557,0.026,0.828,23.407
Arbre de décision (défaut),0.728,0.489,0.512,0.500,0.014,0.660,6.435
Baseline - toujours 'reste',0.735,0.000,0.000,0.000,0.000,0.500,4.892


### 4.2 Arbre de décision pondéré

In [16]:
evaluer("Arbre de décision + class_weight",
        DecisionTreeClassifier(class_weight="balanced", random_state=SEED),
        remarque="pondération des classes, profondeur non limitée")
tableau()

,accuracy_moy,precision_moy,recall_moy,f1_moy,f1_std,roc_auc_moy,temps_s
experience,,,,,,,
Régression logistique + class_weight,0.752,0.521,0.792,0.628,0.023,0.848,6.248
Régression logistique (défaut),0.808,0.673,0.536,0.596,0.033,0.849,6.169
Forêt aléatoire (défaut),0.791,0.638,0.495,0.557,0.026,0.828,23.407
Arbre de décision (défaut),0.728,0.489,0.512,0.500,0.014,0.660,6.435
Arbre de décision + class_weight,0.734,0.498,0.500,0.499,0.024,0.659,6.397
Baseline - toujours 'reste',0.735,0.000,0.000,0.000,0.000,0.500,4.892


### 4.3 Forêt aléatoire pondérée

In [17]:
evaluer("Forêt aléatoire + class_weight",
        RandomForestClassifier(class_weight="balanced", random_state=SEED),
        remarque="pondération des classes, profondeur non limitée")
tableau()

,accuracy_moy,precision_moy,recall_moy,f1_moy,f1_std,roc_auc_moy,temps_s
experience,,,,,,,
Régression logistique + class_weight,0.752,0.521,0.792,0.628,0.023,0.848,6.248
Forêt aléatoire + class_weight,0.780,0.576,0.647,0.610,0.028,0.830,25.153
Régression logistique (défaut),0.808,0.673,0.536,0.596,0.033,0.849,6.169
Forêt aléatoire (défaut),0.791,0.638,0.495,0.557,0.026,0.828,23.407
Arbre de décision (défaut),0.728,0.489,0.512,0.500,0.014,0.660,6.435
Arbre de décision + class_weight,0.734,0.498,0.500,0.499,0.024,0.659,6.397
Baseline - toujours 'reste',0.735,0.000,0.000,0.000,0.000,0.500,4.892


### 4.4 Régression logistique + SMOTE

SMOTE équilibre les classes dans l'entraînement de chaque pli (`sampling_strategy="auto"` : la classe churn est ramenée à la taille de la classe « reste »). La régression logistique est ici utilisée sans `class_weight`, pour ne pas corriger deux fois le même déséquilibre.

In [18]:
from imblearn.over_sampling import SMOTE

evaluer("Régression logistique + SMOTE",
        LogisticRegression(max_iter=1000, random_state=SEED),
        remarque="SMOTE sur les plis d'entraînement",
        sampler=SMOTE(random_state=SEED))
tableau()

,accuracy_moy,precision_moy,recall_moy,f1_moy,f1_std,roc_auc_moy,temps_s
experience,,,,,,,
Régression logistique + class_weight,0.752,0.521,0.792,0.628,0.023,0.848,6.248
Régression logistique + SMOTE,0.762,0.538,0.749,0.626,0.021,0.840,9.397
Forêt aléatoire + class_weight,0.780,0.576,0.647,0.610,0.028,0.830,25.153
Régression logistique (défaut),0.808,0.673,0.536,0.596,0.033,0.849,6.169
Forêt aléatoire (défaut),0.791,0.638,0.495,0.557,0.026,0.828,23.407
Arbre de décision (défaut),0.728,0.489,0.512,0.500,0.014,0.660,6.435
Arbre de décision + class_weight,0.734,0.498,0.500,0.499,0.024,0.659,6.397
Baseline - toujours 'reste',0.735,0.000,0.000,0.000,0.000,0.500,4.892


### 4.5 Forêt aléatoire + SMOTE

In [19]:
evaluer("Forêt aléatoire + SMOTE",
        RandomForestClassifier(random_state=SEED),
        remarque="SMOTE sur les plis d'entraînement",
        sampler=SMOTE(random_state=SEED))
tableau()

,accuracy_moy,precision_moy,recall_moy,f1_moy,f1_std,roc_auc_moy,temps_s
experience,,,,,,,
Régression logistique + class_weight,0.752,0.521,0.792,0.628,0.023,0.848,6.248
Régression logistique + SMOTE,0.762,0.538,0.749,0.626,0.021,0.840,9.397
Forêt aléatoire + class_weight,0.780,0.576,0.647,0.610,0.028,0.830,25.153
Régression logistique (défaut),0.808,0.673,0.536,0.596,0.033,0.849,6.169
Forêt aléatoire + SMOTE,0.786,0.600,0.585,0.592,0.025,0.825,32.294
Forêt aléatoire (défaut),0.791,0.638,0.495,0.557,0.026,0.828,23.407
Arbre de décision (défaut),0.728,0.489,0.512,0.500,0.014,0.660,6.435
Arbre de décision + class_weight,0.734,0.498,0.500,0.499,0.024,0.659,6.397
Baseline - toujours 'reste',0.735,0.000,0.000,0.000,0.000,0.500,4.892


### 4.6 Lecture des résultats

| Modèle | Traitement | Precision | **Recall** | **F1** | ROC-AUC |
|---|---|---|---|---|---|
| Régression logistique | aucun | 0,673 | 0,536 | 0,596 | 0,849 |
| | `class_weight` | 0,521 | **0,792** | **0,628** | 0,848 |
| | SMOTE | 0,538 | 0,749 | 0,626 | 0,840 |
| Forêt aléatoire | aucun | 0,638 | 0,495 | 0,557 | 0,828 |
| | `class_weight` | 0,576 | 0,647 | 0,610 | 0,830 |
| | SMOTE | 0,600 | 0,585 | 0,592 | 0,825 |
| Arbre de décision | aucun | 0,489 | 0,512 | 0,500 | 0,660 |
| | `class_weight` | 0,498 | 0,500 | 0,499 | 0,659 |

**Ce qu'on observe**
1. **Le recall augmente nettement** pour la régression logistique (0,536 → 0,792, soit environ 25 clients sur 100 qui partent détectés en plus) et pour la forêt (0,495 → 0,647 avec la pondération). C'est bien l'effet attendu.
2. **La precision baisse** (0,673 → 0,521 pour la régression logistique) et l'accuracy aussi (0,808 → 0,752) : on signale davantage de clients qui, finalement, ne partiraient pas. C'est le compromis assumé : le projet considère qu'**un client qui part non détecté coûte plus cher qu'une action de fidélisation inutile**.
3. **Le F1 progresse, mais moins que le recall** (+0,03 pour la régression logistique, +0,05 pour la forêt) : le gain en recall est en partie compensé par la perte en precision.
4. **Le ROC-AUC ne bouge presque pas** (0,849 → 0,848). C'est normal : l'AUC mesure la capacité à *classer* les clients du plus au moins à risque, pas le seuil de décision. Pondérer ou rééquilibrer **déplace le seuil** (on déclare « churn » plus facilement) mais n'améliore pas la qualité du classement. Pour progresser sur l'AUC, il faudra de meilleurs réglages (étape 5), pas un rééquilibrage.
5. **`class_weight` et SMOTE se valent** : pour la régression logistique, F1 0,628 contre 0,626, un écart bien inférieur à l'écart-type entre plis (≈ 0,02) ; pour la forêt, `class_weight` fait un peu mieux (0,610 contre 0,592). Comme `class_weight` est plus simple, plus rapide et n'invente pas de clients synthétiques, **on le retient pour la suite**.
6. **L'arbre de décision ne change pas** (0,500 → 0,499) : sans limite de profondeur, il isole presque chaque client dans sa propre feuille, donc les poids n'ont quasiment aucun effet. Son problème reste le sur-apprentissage, à traiter par le réglage de sa profondeur.

**Décision pour l'étape 5** : on garde `class_weight="balanced"` pour les trois modèles et on optimise leurs hyperparamètres (profondeur, régularisation, nombre d'arbres…) en visant le F1 de la classe churn.

**Fin de l'étape 4.** Prochaine étape : optimisation des hyperparamètres.

## 5. Optimisation des hyperparamètres

**Principe.** Les hyperparamètres sont les réglages qu'on fixe *avant* l'entraînement (profondeur d'un arbre, force de la régularisation…). Les valeurs par défaut ne sont pas forcément les meilleures pour nos données. On les cherche en testant plusieurs combinaisons, chacune évaluée par la **même validation croisée** (5 plis, prétraitement inclus dans chaque pli) que depuis le début.

**Choix méthodologiques**
- **Métrique d'optimisation : le F1 de la classe churn** (décision prise à l'étape 2). Les autres métriques sont enregistrées en parallèle.
- **`class_weight="balanced"` conservé** pour les trois modèles (décision de l'étape 4).
- **Recherche en grille** (`GridSearchCV`, toutes les combinaisons) quand l'espace est petit : régression logistique et arbre.
- **Recherche aléatoire** (`RandomizedSearchCV`, un échantillon de combinaisons tirées au hasard) pour la forêt, dont l'espace de réglages est plus grand et chaque essai plus long.
- On enregistre aussi le **F1 sur les plis d'entraînement**, pour vérifier que l'optimisation réduit bien le sur-apprentissage.

**Une précaution à connaître.** Quand on choisit la meilleure combinaison parmi beaucoup d'autres avec le même découpage, son score est légèrement **optimiste** : on a gardé celle qui, par chance, s'en sort le mieux sur ces plis. L'écart est en général petit, mais on ne présentera ces chiffres que comme des scores de **validation**. La vraie mesure de performance se fait sur le jeu de test, une seule fois, avec le modèle final.

### 5.1 Outils de recherche

Deux fonctions : `lancer_recherche` exécute une recherche et enregistre le meilleur résultat dans le même journal `experiences` ; `top_resultats` affiche les meilleures combinaisons testées.

In [20]:
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV


def pipe_avec(modele):
    """Prétraitement neuf + modèle, comme dans evaluer()."""
    return ImbPipeline([("prep", make_preprocessor()), ("modele", modele)])


def lancer_recherche(nom, recherche, remarque=""):
    """Lance la recherche sur le train et enregistre la meilleure combinaison dans le journal."""
    debut = time.time()
    recherche.fit(X_train, y_train)
    duree = time.time() - debut
    r, i = recherche.cv_results_, recherche.best_index_
    ligne = {"experience": nom}
    for m in scoring:
        ligne[f"{m}_moy"] = r[f"mean_test_{m}"][i]
        ligne[f"{m}_std"] = r[f"std_test_{m}"][i]
    ligne["f1_train"] = r["mean_train_f1"][i]
    ligne["temps_s"] = duree
    ligne["parametres"] = str(recherche.best_params_)
    ligne["remarque"] = f"{remarque} | {len(r['params'])} combinaisons testées".strip(" |")
    experiences.append(ligne)
    print(f"{len(r['params'])} combinaisons x {N_SPLITS} plis en {duree:.0f} s")
    print("Meilleurs paramètres :", recherche.best_params_)
    print(f"F1 validation = {ligne['f1_moy']:.3f} | F1 entraînement = {ligne['f1_train']:.3f}")
    return recherche


def top_resultats(recherche, n=8):
    """Les n meilleures combinaisons, avec F1 de validation et d'entraînement."""
    r = pd.DataFrame(recherche.cv_results_)
    cols = [c for c in r.columns if c.startswith("param_")] + ["mean_test_f1", "std_test_f1", "mean_test_recall",
                                                              "mean_test_roc_auc", "mean_train_f1"]
    out = r.sort_values("mean_test_f1", ascending=False)[cols].head(n)
    out.columns = [c.replace("param_modele__", "") for c in out.columns]
    return out.round(3).reset_index(drop=True)

### 5.2 Régression logistique : force de la régularisation `C`

`C` règle la **régularisation** : plus `C` est petit, plus on freine les poids du modèle (modèle plus simple, moins sensible au bruit et aux variables corrélées comme `tenure` / `TotalCharges`) ; plus `C` est grand, plus le modèle colle aux données. La valeur par défaut est `C = 1`. On teste 10 valeurs de 0,001 à 100, sur une échelle logarithmique.

In [21]:
grille_lr = {"modele__C": [0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1, 3, 10, 100]}

recherche_lr = lancer_recherche(
    "Régression logistique optimisée",
    GridSearchCV(pipe_avec(LogisticRegression(max_iter=1000, class_weight="balanced", random_state=SEED)),
                 grille_lr, scoring=scoring, refit="f1", cv=cv, n_jobs=-1, return_train_score=True),
    remarque="grille sur C, class_weight=balanced")
top_resultats(recherche_lr, n=10)

10 combinaisons x 5 plis en 25 s
Meilleurs paramètres : {'modele__C': 0.01}
F1 validation = 0.635 | F1 entraînement = 0.639


,C,mean_test_f1,std_test_f1,mean_test_recall,mean_test_roc_auc,mean_train_f1
0,0.010,0.635,0.021,0.791,0.849,0.639
1,0.030,0.634,0.020,0.795,0.849,0.640
2,0.003,0.632,0.018,0.786,0.847,0.637
3,100.000,0.632,0.024,0.797,0.848,0.639
4,10.000,0.631,0.023,0.795,0.848,0.640
5,3.000,0.630,0.024,0.795,0.849,0.640
6,0.001,0.629,0.022,0.779,0.845,0.633
7,0.100,0.629,0.020,0.794,0.849,0.640
8,0.300,0.629,0.022,0.793,0.848,0.640
9,1.000,0.628,0.023,0.792,0.848,0.640


### 5.3 Arbre de décision : limiter la complexité

C'est le modèle qui sur-apprenait le plus (F1 de 0,997 sur le train contre 0,50 en validation). On bride sa croissance avec :
- `max_depth` : nombre maximal de questions successives ;
- `min_samples_leaf` : nombre minimal de clients dans une feuille (évite les feuilles qui ne décrivent qu'un ou deux clients) ;
- `criterion` : mesure de qualité d'une question (`gini` ou `entropy`).

8 profondeurs × 6 tailles de feuille × 2 critères = 96 combinaisons.

In [22]:
grille_arbre = {
    "modele__max_depth": [2, 3, 4, 5, 6, 8, 10, None],
    "modele__min_samples_leaf": [1, 5, 10, 20, 50, 100],
    "modele__criterion": ["gini", "entropy"],
}

recherche_arbre = lancer_recherche(
    "Arbre de décision optimisé",
    GridSearchCV(pipe_avec(DecisionTreeClassifier(class_weight="balanced", random_state=SEED)),
                 grille_arbre, scoring=scoring, refit="f1", cv=cv, n_jobs=-1, return_train_score=True),
    remarque="grille sur max_depth, min_samples_leaf, criterion; class_weight=balanced")
top_resultats(recherche_arbre, n=10)

96 combinaisons x 5 plis en 94 s
Meilleurs paramètres : {'modele__criterion': 'gini', 'modele__max_depth': 10, 'modele__min_samples_leaf': 50}
F1 validation = 0.621 | F1 entraînement = 0.652


,criterion,max_depth,min_samples_leaf,mean_test_f1,std_test_f1,mean_test_recall,mean_test_roc_auc,mean_train_f1
0,gini,10,50,0.621,0.014,0.795,0.832,0.652
1,gini,None,50,0.621,0.014,0.795,0.832,0.652
2,gini,8,50,0.620,0.014,0.799,0.833,0.651
3,gini,6,50,0.620,0.016,0.794,0.833,0.646
4,gini,3,100,0.619,0.018,0.776,0.815,0.622
5,gini,3,1,0.619,0.018,0.776,0.815,0.622
6,gini,3,20,0.619,0.018,0.776,0.815,0.622
7,gini,3,5,0.619,0.018,0.776,0.815,0.622
8,entropy,3,1,0.619,0.018,0.776,0.816,0.622
9,entropy,3,5,0.619,0.018,0.776,0.816,0.622


### 5.4 Forêt aléatoire : recherche aléatoire

La forêt a plus de réglages et chaque essai coûte plus cher (des centaines d'arbres à entraîner × 5 plis). Tester toutes les combinaisons serait trop long : on tire donc **25 combinaisons au hasard** dans l'espace ci-dessous. Cette approche (`RandomizedSearchCV`) trouve en général des réglages presque aussi bons qu'une grille complète pour une fraction du temps.

| Réglage | Rôle | Valeurs testées |
|---|---|---|
| `n_estimators` | nombre d'arbres | 100, 200, 300 |
| `max_depth` | profondeur maximale de chaque arbre | 4, 6, 8, 12, illimitée |
| `min_samples_leaf` | taille minimale d'une feuille | 1, 2, 5, 10, 20, 50 |
| `max_features` | nombre de variables examinées à chaque question | `sqrt`, `log2`, 30 %, 50 % |
| `class_weight` | pondération du churn | `balanced`, `balanced_subsample` (recalculée pour chaque arbre) |

In [23]:
distributions_rf = {
    "modele__n_estimators": [100, 200, 300],
    "modele__max_depth": [4, 6, 8, 12, None],
    "modele__min_samples_leaf": [1, 2, 5, 10, 20, 50],
    "modele__max_features": ["sqrt", "log2", 0.3, 0.5],
    "modele__class_weight": ["balanced", "balanced_subsample"],
}

recherche_rf = lancer_recherche(
    "Forêt aléatoire optimisée",
    RandomizedSearchCV(pipe_avec(RandomForestClassifier(random_state=SEED)),
                       distributions_rf, n_iter=25, scoring=scoring, refit="f1", cv=cv,
                       n_jobs=-1, random_state=SEED, return_train_score=True),
    remarque="recherche aléatoire, 25 tirages")
top_resultats(recherche_rf, n=10)

25 combinaisons x 5 plis en 134 s
Meilleurs paramètres : {'modele__n_estimators': 100, 'modele__min_samples_leaf': 2, 'modele__max_features': 'log2', 'modele__max_depth': 6, 'modele__class_weight': 'balanced_subsample'}
F1 validation = 0.634 | F1 entraînement = 0.654


,n_estimators,min_samples_leaf,max_features,max_depth,class_weight,mean_test_f1,std_test_f1,mean_test_recall,mean_test_roc_auc,mean_train_f1
0,100,2,log2,6,balanced_subsample,0.634,0.013,0.801,0.847,0.654
1,200,10,0.3,6,balanced_subsample,0.634,0.016,0.779,0.846,0.666
2,300,20,sqrt,6,balanced_subsample,0.633,0.014,0.797,0.847,0.649
3,200,1,sqrt,8,balanced,0.633,0.012,0.780,0.846,0.721
4,200,50,sqrt,6,balanced_subsample,0.632,0.015,0.801,0.847,0.642
5,100,50,0.3,12,balanced_subsample,0.631,0.014,0.792,0.847,0.649
6,100,10,0.5,4,balanced,0.631,0.016,0.789,0.843,0.637
7,300,20,sqrt,6,balanced,0.631,0.014,0.801,0.847,0.647
8,200,50,0.3,None,balanced,0.630,0.017,0.796,0.846,0.645
9,100,1,0.5,4,balanced,0.630,0.015,0.788,0.843,0.637


### 5.5 Bilan de l'optimisation : avant / après

In [24]:
avant_apres = pd.DataFrame(experiences).set_index("experience")
paires = {
    "Régression logistique": ("Régression logistique + class_weight", "Régression logistique optimisée"),
    "Arbre de décision": ("Arbre de décision + class_weight", "Arbre de décision optimisé"),
    "Forêt aléatoire": ("Forêt aléatoire + class_weight", "Forêt aléatoire optimisée"),
}
lignes = []
for modele, (avant, apres) in paires.items():
    for etat, cle in [("avant", avant), ("après", apres)]:
        r = avant_apres.loc[cle]
        lignes.append({"modèle": modele, "état": etat, "precision": r["precision_moy"], "recall": r["recall_moy"],
                       "F1": r["f1_moy"], "F1 ± écart-type": f"{r['f1_moy']:.3f} ± {r['f1_std']:.3f}",
                       "ROC-AUC": r["roc_auc_moy"], "F1 train": r.get("f1_train", float("nan"))})
pd.DataFrame(lignes).set_index(["modèle", "état"]).drop(columns="F1").round(3)

precision  recall F1 ± écart-type  ROC-AUC  \
modèle                état                                                
Régression logistique avant      0.521   0.792   0.628 ± 0.023    0.848   
                      après      0.530   0.791   0.635 ± 0.021    0.849   
Arbre de décision     avant      0.498   0.500   0.499 ± 0.024    0.659   
                      après      0.510   0.795   0.621 ± 0.014    0.832   
Forêt aléatoire       avant      0.576   0.647   0.610 ± 0.028    0.830   
                      après      0.525   0.801   0.634 ± 0.013    0.847   

                             F1 train  
modèle                état             
Régression logistique avant       NaN  
                      après     0.639  
Arbre de décision     avant       NaN  
                      après     0.652  
Forêt aléatoire       avant       NaN  
                      après     0.654

### 5.6 Lecture des résultats

| Modèle | Meilleurs réglages | F1 avant → après | Recall | ROC-AUC | F1 train → validation |
|---|---|---|---|---|---|
| Régression logistique | `C = 0,01` | 0,628 → **0,635** | 0,791 | 0,849 | 0,639 → 0,635 |
| Arbre de décision | `max_depth = 10`, `min_samples_leaf = 50`, `gini` | 0,499 → **0,621** | 0,795 | 0,832 | 0,652 → 0,621 |
| Forêt aléatoire | 100 arbres, `max_depth = 6`, `min_samples_leaf = 2`, `max_features = log2`, `balanced_subsample` | 0,610 → **0,634** | 0,801 | 0,847 | 0,654 → 0,634 |

**Ce qu'on observe**
1. **L'optimisation a surtout servi à corriger le sur-apprentissage de l'arbre.** Son F1 passe de 0,499 à 0,621 et son ROC-AUC de 0,659 à 0,832. L'écart entre le F1 d'entraînement et de validation tombe de 0,50 (avec les réglages par défaut) à 0,03 : en l'obligeant à garder au moins 50 clients par feuille, il apprend des règles générales au lieu de mémoriser des cas particuliers.
2. **La forêt progresse nettement** (F1 0,610 → 0,634, recall 0,647 → 0,801) en limitant la profondeur des arbres à 6. Son écart train/validation passe de 0,44 à 0,02.
3. **La régression logistique gagne très peu** (+0,007) : elle était déjà bien réglée et ne sur-apprenait pas. Les valeurs de `C` testées donnent toutes un F1 entre 0,628 et 0,635 : le choix de `C` n'a pas d'effet réel ici.
4. **Les trois modèles se retrouvent au même niveau** : F1 entre 0,621 et 0,635, recall autour de 0,79–0,80. Les écarts entre eux (≤ 0,014) sont **du même ordre que la variation d'un pli à l'autre** (écart-type de 0,013 à 0,021). On ne peut donc **pas affirmer** qu'un de ces modèles est meilleur qu'un autre sur ce critère.
5. **Une limite semble atteinte avec ces variables** : des modèles aussi différents (linéaire, arbre, ensemble d'arbres) plafonnent au même niveau, ce qui suggère que l'information disponible dans les données, plutôt que le choix de l'algorithme, limite la performance.
6. Les meilleures combinaisons testées forment un **plateau** : dans la forêt, les 10 meilleures vont de 0,631 à 0,634. Les réglages exacts retenus ne sont donc pas critiques ; beaucoup d'autres auraient donné un résultat équivalent. Pour l'arbre, un arbre de profondeur 3 obtient 0,619, presque autant, avec une structure beaucoup plus lisible.

**Rappel de prudence.** Ces scores sont des scores de validation de la combinaison *choisie comme la meilleure* : ils sont légèrement optimistes. L'évaluation honnête sur le jeu de test (jamais utilisé jusqu'ici) est le travail de la partie 5.

**Fin de l'étape 5.** Prochaine étape : tableau comparatif final de toutes les expériences et sauvegarde des modèles.

## 6. Tableau comparatif final et sauvegarde des modèles

Cette dernière section de la partie 4 :
1. rassemble **toutes les expériences** dans un tableau comparatif, exporté dans `reports/` ;
2. **sauvegarde les trois modèles optimisés** dans `models/`, pour que la partie 5 puisse les évaluer sans tout ré-entraîner.

### 6.1 Tableau comparatif de toutes les expériences

Une ligne par expérience, triée par F1 décroissant : référence, modèles par défaut, avec pondération, avec SMOTE, et modèles optimisés. Les colonnes `_std` donnent la variation d'un pli à l'autre (plus elle est grande, moins l'écart entre deux modèles est fiable). L'export CSV garde en plus les **paramètres** de chaque expérience, comme demandé par le TP.

In [25]:
journal = pd.DataFrame(experiences).set_index("experience").sort_values("f1_moy", ascending=False)

colonnes = ["accuracy_moy", "precision_moy", "recall_moy", "f1_moy", "f1_std", "roc_auc_moy", "roc_auc_std", "f1_train"]
tableau_final = journal[colonnes].rename(columns={
    "accuracy_moy": "accuracy", "precision_moy": "precision", "recall_moy": "recall", "f1_moy": "F1",
    "f1_std": "F1 écart-type", "roc_auc_moy": "ROC-AUC", "roc_auc_std": "AUC écart-type", "f1_train": "F1 train"})

# Export : tableau complet, avec les paramètres et remarques de chaque expérience
REPORTS_DIR.mkdir(exist_ok=True)
export = journal[colonnes + ["temps_s", "parametres", "remarque"]].round(4)
export.to_csv(REPORTS_DIR / "tableau_comparatif_experiences.csv", encoding="utf-8")
print(f"{len(export)} expériences exportées dans {REPORTS_DIR / 'tableau_comparatif_experiences.csv'}")

tableau_final.round(3)

12 expériences exportées dans ..\reports\tableau_comparatif_experiences.csv


,accuracy,precision,recall,F1,F1 écart-type,ROC-AUC,AUC écart-type,F1 train
experience,,,,,,,,
Régression logistique optimisée,0.759,0.530,0.791,0.635,0.021,0.849,0.011,0.639
Forêt aléatoire optimisée,0.755,0.525,0.801,0.634,0.013,0.847,0.010,0.654
Régression logistique + class_weight,0.752,0.521,0.792,0.628,0.023,0.848,0.013,NaN
Régression logistique + SMOTE,0.762,0.538,0.749,0.626,0.021,0.840,0.014,NaN
Arbre de décision optimisé,0.743,0.510,0.795,0.621,0.014,0.832,0.014,0.652
Forêt aléatoire + class_weight,0.780,0.576,0.647,0.610,0.028,0.830,0.009,NaN
Régression logistique (défaut),0.808,0.673,0.536,0.596,0.033,0.849,0.013,NaN
Forêt aléatoire + SMOTE,0.786,0.600,0.585,0.592,0.025,0.825,0.008,NaN
Forêt aléatoire (défaut),0.791,0.638,0.495,0.557,0.026,0.828,0.009,NaN


### 6.2 Sauvegarde des modèles optimisés

`GridSearchCV` / `RandomizedSearchCV` ont déjà **ré-entraîné la meilleure combinaison sur l'ensemble du train** (`refit`). Chaque `best_estimator_` est un pipeline complet : **prétraitement + modèle**. On peut donc lui donner directement des clients bruts (tels que dans `telco_clean_features.csv`, sans `customerID` ni `Churn`) et obtenir une prédiction.

On sauvegarde avec `joblib`. Un fichier ne se recharge correctement qu'avec les **mêmes versions** de scikit-learn et imbalanced-learn (fixées dans `requirements.txt`).

In [26]:
MODELS_DIR.mkdir(exist_ok=True)

def sauvegarder(nom_fichier, recherche):
    chemin = MODELS_DIR / nom_fichier
    joblib.dump(recherche.best_estimator_, chemin, compress=3)
    print(f"{chemin.name:28s} {chemin.stat().st_size / 1024:8.1f} Ko | paramètres : {recherche.best_params_}")

sauvegarder("logreg_optimisee.joblib", recherche_lr)

logreg_optimisee.joblib           4.2 Ko | paramètres : {'modele__C': 0.01}
